# Project 4 Live Demo：推論 + 控制（可切換模型）

流程：`camera.py` 取像 → `preprocessing.py`（把車道線標成紅色，和訓練時相同，固定開啟）→ `loader.py` 載入模型（`resnet18.py` / `mobilenet_v2.py`）並推論 → `control.py` 轉向角 + PID + 差速控制。畫面顯示的就是模型實際看到的影像。

- 從下拉選單選模型後，**載入完成就開始推論**，開不開車都會持續推論、顯示。
- **「開始行駛」按鈕**：按一下開始送馬達（按鈕變紅色「停止」），再按一下停車。安全機制停車時，按鈕會自動變回綠色，並顯示停車原因。
- 載入模型時會用 `.json` 裡存的畫面做 **sanity 比對**（TensorRT／FP16 轉換是否正確）；比對沒通過**禁止開車**，但推論照常顯示。
- 安全機制（開車時）：模型輸出 NaN／超出畫面、目標點太近、相機沒有新畫面、推論或相機逾時、推論出錯 → 立刻停車。
- 所有控制選項都是按鈕或滑桿，改了立刻生效。
- **沒有車也能開**：偵測不到 JetBot 就改用假馬達（只顯示數值）；鏡頭打不開就顯示「未連接鏡頭」。

下拉選單**只列 `models_trt/` 的 TensorRT 模型**。流程：把 PC 上 `train.py` 產生的 `.pth` + `.json`（要用 trtexec 的話加 `.onnx`）上傳到 `models/`，在車上執行 `python3 convert_trt.py` 轉換，結果放在 `models_trt/`。沒有 GPU 的電腦載入不了 TRT 模型，但介面照樣可以看。

**推論和畫面是兩個獨立的執行緒**（`drive.py`）：推論＋控制執行緒只做讀圖、推論、控制、送馬達、寫 CSV，結果放進只保留最新一筆的信箱（`latest_slot.py`）；顯示執行緒自己去拿最新的一筆來畫圖、壓 JPEG、更新介面。Wi-Fi 慢的時候只有畫面變慢或跳幀（顯示 FPS 和 JPEG 品質會自動降低，網路恢復後再慢慢調回來），控制週期不受影響。stats 列會顯示控制週期 p50／p95、各段耗時、顯示 FPS 和 JPEG 品質。

## 1. 設定

In [ ]:
import os, sys, time, threading, csv, datetime, html, traceback
import numpy as np
import torch
import ipywidgets as widgets
from IPython.display import display

# Make the modules in project4/ importable
sys.path.insert(0, os.path.abspath('.'))
# Camera sources, tensor conversion, input disturbance and drawing helpers
import camera as cam_mod
# Checkpoint loading (PyTorch / TensorRT) and listing for the dropdown
from loader import load_model, list_models
# Lane marking: the same preprocessing the models were trained with
from preprocessing import apply_frame as mark_lanes
# Steering controller and its default settings
from control import SteeringController, DEFAULTS as CTRL
# Robot-side comparison with the outputs stored at training time
import sanity
# Git commit of the running code, compared with the model's
import version
# Decoupled inference/control loop and display thread
from drive import DriveLoop, DisplayWorker

# Folder with the TensorRT models made by convert_trt.py (the only ones offered here)
MODELS_DIR = 'models_trt'
# Frame source: 'jetbot' | 'csi' (never queues old frames) | 'opencv' (webcam) | 'folder' | 'dummy'
CAMERA_SOURCE = 'jetbot'
# Image folder replayed when CAMERA_SOURCE = 'folder'
CAMERA_FOLDER = 'data/dataset_xy_2026-10-21/dataset_xy'
# Folder for the CSV drive logs
LOG_DIR = 'runs'
# Drop frames queued during a model switch (jetbot source restarts its pipeline, ~1-2 s)
FLUSH_ON_SWITCH = True
# Max display updates per second; higher rates make the Wi-Fi / browser fall behind
DISPLAY_FPS = 15

# Run on the GPU when there is one
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Show the device and the checkpoints that were found
print('device:', device, '| models:', [os.path.basename(p) for _, p in list_models(MODELS_DIR)])

## 2. 相機與車子（偵測不到就改用假的，沒有車也能看介面）

In [ ]:
# Stand-in for a jetbot motor when no robot is attached
class _FakeMotor(object):
    # Motor value written by the control loop
    value = 0.0

# Stand-in for jetbot.Robot with the same attributes the loop uses
class _FakeRobot(object):
    # Create two fake motors
    def __init__(self):
        # Left and right motor
        self.left_motor, self.right_motor = _FakeMotor(), _FakeMotor()
    # Set both motors to zero
    def stop(self):
        # Stop both motors
        self.left_motor.value = self.right_motor.value = 0.0

# Try the real robot first
try:
    from jetbot import Robot
    # Real motor driver
    robot = Robot()
    # Status text for the UI
    robot_text = '車子：已連接'
# No jetbot package or no motor driver: simulate it
except Exception as e:
    # Fake motors (values are only displayed)
    robot = _FakeRobot()
    # Status text with the reason
    robot_text = '車子：<b>未連接</b>（馬達值只顯示）— %s' % html.escape(repr(e)[:120])
# Make sure the robot starts standing still
robot.stop()

# Try to open the chosen frame source
try:
    # Camera object with read() / flush() / stop()
    camera = cam_mod.open_camera(CAMERA_SOURCE, folder=CAMERA_FOLDER)
    # Status text for the UI
    camera_text = '鏡頭：已連接（%s）' % CAMERA_SOURCE
# No camera (e.g. on a PC without jetbot): run without frames
except Exception as e:
    # No frames at all
    camera = None
    # Status text with the reason
    camera_text = '鏡頭：<b>未連接</b> — %s' % html.escape(repr(e)[:120])

# Controller with the default settings; the sliders overwrite them every frame
controller = SteeringController()
# Show what was found
print(robot_text.replace('<b>', '').replace('</b>', ''))
# Same for the camera
print(camera_text.replace('<b>', '').replace('</b>', ''))

## 3. 介面

In [ ]:
# Common layout for the parameter sliders
S = dict(layout=widgets.Layout(width='330px'), style={'description_width': '120px'})
# Common settings for the read-only value displays
R = dict(disabled=True, readout_format='.3f', **S)
# Common layout for the toggle buttons
T = dict(layout=widgets.Layout(width='160px'))

# --- Model ----------------------------------------------------------------
# Model selector filled with the checkpoints in MODELS_DIR; nothing selected at start
model_dd = widgets.Dropdown(description='模型', options=list_models(MODELS_DIR), value=None,
                            layout=widgets.Layout(width='520px'), style={'description_width': '50px'})
# Button to rescan MODELS_DIR
refresh_btn = widgets.Button(description='重新掃描', layout=widgets.Layout(width='90px'))
# Information line about the loaded model (and errors)
model_info = widgets.HTML(value='<b>未載入模型</b>：請從上面的下拉選單選擇')
# Result of the sanity comparison
sanity_info = widgets.HTML(value='')
# Robot and camera connection state
hw_info = widgets.HTML(value='%s ｜ %s' % (robot_text, camera_text))

# --- Picture --------------------------------------------------------------
# Live image (what the model sees, with the prediction drawn on it)
image_w = widgets.Image(format='jpeg', width=336, height=336)
# Placeholder shown instead of the picture when there is no camera
no_cam = widgets.HTML(value=('<div style="width:336px;height:336px;background:#333;color:#eee;display:flex;'
                             'align-items:center;justify-content:center;font-size:22px">未連接鏡頭</div>'))
# Show the placeholder only without a camera
(image_w if camera is None else no_cam).layout.display = 'none'

# --- Control settings (defaults from control.DEFAULTS) ---------------------
# Steering-angle formula
formula_tb = widgets.ToggleButtons(options=[('隊友公式', 'teammate'), ('投影片公式', 'slides')],
                                   value=CTRL['formula'], description='轉向角公式', style={'description_width': '120px'})
# Top speed on straights
max_speed_s = widgets.FloatSlider(min=0.0, max=1.0, step=0.01, value=CTRL['max_speed'], description='max speed', **S)
# Lowest speed in sharp turns
min_speed_s = widgets.FloatSlider(min=0.0, max=1.0, step=0.01, value=CTRL['min_speed'], description='min speed', **S)
# Proportional steering gain
kp_s = widgets.FloatSlider(min=0.0, max=1.0, step=0.01, value=CTRL['kp'], description='kp', **S)
# Integral steering gain
ki_s = widgets.FloatSlider(min=0.0, max=0.5, step=0.005, value=CTRL['ki'], description='ki',
                           readout_format='.3f', **S)
# Derivative steering gain (per second)
kd_s = widgets.FloatSlider(min=0.0, max=0.1, step=0.001, value=CTRL['kd'], description='kd (每秒)',
                           readout_format='.3f', **S)
# Steering offset that compensates for unequal motors
bias_s = widgets.FloatSlider(min=-0.3, max=0.3, step=0.01, value=CTRL['bias'], description='steering bias', **S)
# How much the speed drops in sharp turns
slow_s = widgets.FloatSlider(min=0.0, max=1.0, step=0.05, value=CTRL['turn_slowdown'], description='轉彎減速', **S)
# Low-pass filter on the angle (1 = off)
alpha_s = widgets.FloatSlider(min=0.05, max=1.0, step=0.05, value=CTRL['angle_alpha'], description='角度平滑 α', **S)
# Largest motor change per second (0 = off)
slew_s = widgets.FloatSlider(min=0.0, max=3.0, step=0.1, value=CTRL['slew_per_second'], description='馬達變化率/秒', **S)
# Largest motor value
maxout_s = widgets.FloatSlider(min=0.1, max=1.0, step=0.05, value=CTRL['max_output'], description='馬達上限', **S)
# Allow the inner wheel to spin backwards
reverse_tb = widgets.ToggleButton(description='內側輪可反轉', value=CTRL['allow_reverse'], **T)
# Stop when no successful control step happened for this long while driving
timeout_s = widgets.FloatSlider(min=0.2, max=2.0, step=0.1, value=0.5, description='安全逾時 (秒)', **S)

# --- Live readings --------------------------------------------------------
# Predicted x (normalised)
x_r = widgets.FloatSlider(min=-1.25, max=1.25, description='x', **R)
# Predicted y (normalised)
y_r = widgets.FloatSlider(min=-1.25, max=1.25, description='y', **R)
# Filtered steering angle in radians
angle_r = widgets.FloatSlider(min=-3.2, max=3.2, description='angle (rad)', **R)
# Total steering value
steer_r = widgets.FloatSlider(min=-1.0, max=1.0, description='steering', **R)
# Speed actually used
speed_r = widgets.FloatSlider(min=0.0, max=1.0, description='實際 speed', **R)
# Left motor value
left_r = widgets.FloatSlider(min=-1.0, max=1.0, description='left motor', **R)
# Right motor value
right_r = widgets.FloatSlider(min=-1.0, max=1.0, description='right motor', **R)
# FPS, latency and P / I / D terms
stats = widgets.HTML()

# --- Operation ------------------------------------------------------------
# Drive button: green "start" while standing, red "stop" while driving
drive_btn = widgets.Button(description='開始行駛', button_style='success', icon='play',
                           layout=widgets.Layout(width='160px', height='40px'))
# Driving state or the reason of the last stop
drive_status = widgets.HTML(value='停車中')
# Record every control step to a CSV file
record_tb = widgets.ToggleButton(description='錄製 CSV', value=False, icon='circle', **T)

# Arrange all widgets: model rows, picture + controls, operation row
ui = widgets.VBox([
    widgets.HBox([model_dd, refresh_btn]), model_info, sanity_info, hw_info,
    widgets.HBox([
        widgets.VBox([image_w, no_cam,
                      widgets.HBox([drive_btn, record_tb]), drive_status]),
        widgets.VBox([widgets.HTML('<b>控制參數</b>'), formula_tb, max_speed_s, min_speed_s, kp_s, ki_s, kd_s,
                      bias_s, slow_s, alpha_s, slew_s, maxout_s, reverse_tb, timeout_s]),
        widgets.VBox([widgets.HTML('<b>即時讀數</b>'), x_r, y_r, angle_r, steer_r, speed_r, left_r, right_r, stats]),
    ]),
])
# Show the interface
display(ui)

## 4. 推論 / 控制迴圈（執行後就在背景一直跑）

兩個執行緒：`DriveLoop`（推論＋控制＋送馬達，不碰任何 widget）和 `DisplayWorker`（拿最新結果畫圖、更新介面；慢了就跳幀、降低 FPS 和 JPEG 品質）。

In [ ]:
# Readable texts for the loop's stop reasons (codes from drive.STOP_CODES) and statuses
REASONS = {'manual': '手動停止', 'model_switch': '切換模型', 'camera_stalled': '相機超過 %(seconds).1f 秒沒有新畫面',
           'watchdog': '推論或相機逾時（超過 %(seconds).1f 秒）', 'inference_error': '推論錯誤：%(error)s',
           'loop_error': '迴圈錯誤：%(error)s', 'loop_ended': '迴圈結束', 'closed': '結束', 'benchmark': 'benchmark',
           'invalid_prediction': '模型輸出 NaN / inf', 'out_of_range': '模型輸出超出畫面範圍',
           'target_too_close': '目標點太靠近車頭', 'ok': 'ok', 'no_model': '未載入模型'}
# Readable texts for refusals to start driving
REFUSALS = {'no_camera': '未連接鏡頭，無法開車', 'no_model': '未載入模型，無法開車',
            'sanity_failed': 'sanity 比對沒通過，禁止開車'}

# Reason code + details -> readable text
def reason_text(code, detail=None):
    # Text template for the code (unknown codes are shown as they are)
    text = REASONS.get(code, code)
    # Fill in the details (seconds, error) when the template has placeholders
    try:
        # Format with the detail dict
        return text % (detail or {})
    # Missing detail: show the template without it
    except (KeyError, TypeError, ValueError):
        # Unformatted text
        return text

# HTML summary of a loaded model for the info line
def describe(m):
    # Metadata from the checkpoint's .json
    meta = m.meta or {}
    # Start with file name, runtime and dtype
    parts = ['<b>%s</b>' % m.describe()]
    # Add the label (network | augmentation | compression)
    if meta.get('label'): parts.append(meta['label'])
    # Preprocessing the model was trained with
    pre = meta.get('preprocessing') or {}
    # Trained on plain images (train.py --no-lanes): frames are fed without lane marking
    if pre.get('method') == 'none':
        # Orange note
        parts.append('<span style="color:darkorange">不標車道線（--no-lanes 模型）</span>')
    # Unknown preprocessing (very old models): warn
    elif pre.get('method') != 'lane_marking':
        # Red warning text
        parts.append('<span style="color:red">這個模型不是用車道線標示的資料訓練的</span>')
    # Code version the model was trained with vs the code running here
    status, text = version.compare(meta.get('code_version'))
    # Green when identical, orange otherwise
    parts.append('<span style="color:%s">%s</span>' % ('green' if status == 'same' else 'darkorange', html.escape(text)))
    # Parameter count
    if meta.get('params'): parts.append('params %.2fM' % (meta['params'] / 1e6))
    # Multiply-accumulate count
    if meta.get('macs'): parts.append('MACs %.2fG' % (meta['macs'] / 1e9))
    # Validation error: mean pixel distance if recorded, otherwise MSE
    if meta.get('val_metrics'): parts.append('val %.1f px' % meta['val_metrics']['mean_px'])
    # Older models only have the validation MSE
    elif meta.get('best_val_loss') is not None: parts.append('val loss %.4f' % meta['best_val_loss'])
    # Test error (only for models trained on split folders)
    if meta.get('test_metrics'): parts.append('test %.1f px (p95 %.1f)' % (meta['test_metrics']['mean_px'],
                                                                          meta['test_metrics']['p95_px']))
    # One line separated by bars
    return ' | '.join(parts)

# Show the drive button in its standing / driving look
def show_drive_button(driving):
    # Driving: red stop button
    if driving:
        # Red, "stop" text and icon
        drive_btn.description, drive_btn.button_style, drive_btn.icon = '停止', 'danger', 'stop'
    # Standing: green start button
    else:
        # Green, "start" text and icon
        drive_btn.description, drive_btn.button_style, drive_btn.icon = '開始行駛', 'success', 'play'

# Inference + control loop (its own thread; never touches a widget)
drive = DriveLoop(camera, robot, controller, device, log_dir=LOG_DIR, flush_on_switch=FLUSH_ON_SWITCH,
                  timeout=timeout_s.value)

# Show the loop's events (runs in the display thread, never in the loop thread)
def on_events(events):
    # Oldest first
    for ev in events:
        # Event kind
        kind = ev['kind']
        # Loading started
        if kind == 'model_loading':
            # Show progress and clear the old check result
            model_info.value, sanity_info.value = '載入中… %s' % html.escape(os.path.basename(ev['path'])), ''
        # Loading failed
        elif kind == 'model_failed':
            # Red error text
            model_info.value = '<span style="color:red"><b>載入失敗：</b>%s</span>' % html.escape(ev['error'])
        # Emptying the camera queue after loading
        elif kind == 'flushing':
            # Show progress
            model_info.value = '清空相機佇列… %s' % html.escape(os.path.basename(ev['path']))
        # New model active
        elif kind == 'model_loaded':
            # Sanity result of the new model
            result = ev['sanity']
            # Colour by result: green passed, red failed, orange missing
            color = {'passed': 'green', 'failed': 'red'}.get(result['status'], 'darkorange')
            # Text by result
            word = {'passed': 'sanity 比對通過', 'failed': 'sanity 比對沒通過，禁止開車'}.get(result['status'], 'sanity 未比對')
            # Show the result
            sanity_info.value = '<span style="color:%s"><b>%s</b></span>：%s' % (color, word, html.escape(result['message']))
            # Show the model summary
            model_info.value = describe(ev['model'])
        # Inference or loop error: show the traceback
        elif kind in ('inference_error', 'loop_error'):
            # Title by kind
            title = '推論錯誤，模型已卸載' if kind == 'inference_error' else '迴圈錯誤'
            # Red title and the traceback
            model_info.value = ('<span style="color:red"><b>%s：%s</b></span><pre style="font-size:11px">%s</pre>'
                                % (title, html.escape(ev['error']), html.escape(ev['traceback'])))
        # Driving started
        elif kind == 'drive_started':
            # Red stop button
            show_drive_button(True)
            # Driving state
            drive_status.value = '<span style="color:red"><b>行駛中</b></span>'
        # Driving stopped (button, safety check, model switch, ...)
        elif kind == 'drive_stopped':
            # Green start button
            show_drive_button(False)
            # Reset the motor readings
            left_r.value = right_r.value = 0.0
            # Readable reason
            text = html.escape(reason_text(ev['code'], ev.get('detail')))
            # Faults in red, normal stops in plain text
            drive_status.value = ('<span style="color:red"><b>已停車：%s</b></span>' % text if ev['fault']
                                  else '停車中：%s' % text)
        # CSV log opened
        elif kind == 'recording_started':
            # Show where the log goes
            drive_status.value += ' ｜ 錄製到 %s' % html.escape(ev['path'])

# Statistics line: control timing (TASKS M0-1) and display state
def stats_html(status=None, c=None):
    # Loop and display statistics (replaced as a whole by their threads)
    s, d = drive.stats, drive.display_stats
    # Format a number or show a dash
    f = lambda v, fmt='%.1f': (fmt % v) if v is not None else '-'
    # Control: steps per second, period percentiles, stage times
    text = ('控制 <b>%s FPS</b> | 週期 p50 %s / <b>p95 %s ms</b> | 讀圖 %s・前處理 %s・推論 %s・控制 %s ms'
            % (f(s.get('fps')), f(s.get('period_p50_ms')), f(s.get('period_p95_ms')), f(s.get('read_ms'), '%.2f'),
               f(s.get('prep_ms')), f(s.get('infer_ms')), f(s.get('control_ms'), '%.2f')))
    # Display: actual / target FPS, JPEG quality, last render time, skipped results
    text += ('<br>顯示 <b>%s FPS</b>（目標 %s）| JPEG %s | 畫一張 %s ms | 跳過 %s 筆/秒'
             % (f(d.get('fps')), f(d.get('target_fps')), d.get('quality', '-'), f(d.get('render_ms'), '%.0f'),
                f(d.get('skipped_per_s'), '%.0f')))
    # PID terms and controller status when a model runs
    if c is not None:
        # Terms and readable status
        text += '<br>P %.3f I %.3f D %.3f | 狀態 %s' % (c['p'], c['i'], c['d'], html.escape(reason_text(status)))
    # The finished line
    return text

# Draw one loop result into the widgets (runs in the display thread)
def render(item, quality):
    # Lane-marked frame the model saw
    frame = item['frame']
    # No model yet: picture without prediction
    if item['xy'] is None:
        # Show the frame
        image_w.value = cam_mod.to_jpeg(frame, quality)
        # Timing only
        stats.value = stats_html()
        # Done
        return
    # Prediction and control values of that step
    xy, c = item['xy'], item['ctrl']
    # Show x and y (clamped to the slider range)
    x_r.value, y_r.value = max(-1.25, min(1.25, float(xy[0]))), max(-1.25, min(1.25, float(xy[1])))
    # Show angle and (clamped) steering
    angle_r.value, steer_r.value = c['angle'], max(-1.0, min(1.0, c['steering']))
    # Show the speed
    speed_r.value = c['speed']
    # Motor readings (zero while standing)
    left_r.value, right_r.value = item['left'], item['right']
    # Lane-marked frame with prediction and steering bar
    image_w.value = cam_mod.to_jpeg(cam_mod.draw_overlay(frame, xy, c['steering']), quality)
    # Timing, display state, PID terms and status
    stats.value = stats_html(item['status'], c)

# Display thread: newest result only, adapts FPS / JPEG quality when rendering is slow
display_worker = DisplayWorker(drive.slot, render, drive.drain_events, on_events, max_fps=DISPLAY_FPS,
                               stats=drive.display_stats)

# Copy every control widget into the controller
def apply_controls(_=None):
    # Steering formula and speeds
    controller.formula, controller.max_speed, controller.min_speed = formula_tb.value, max_speed_s.value, min_speed_s.value
    # PID gains and bias
    controller.kp, controller.ki, controller.kd, controller.bias = kp_s.value, ki_s.value, kd_s.value, bias_s.value
    # Turn slowdown, angle filter and motor slew limit
    controller.turn_slowdown, controller.angle_alpha, controller.slew_per_second = slow_s.value, alpha_s.value, slew_s.value
    # Motor limit and reverse option
    controller.max_output, controller.allow_reverse = maxout_s.value, reverse_tb.value
    # Safety timeout used by the loop and its watchdog
    drive.timeout = timeout_s.value

# Drive button handler: toggle between start and stop
def on_drive(_=None):
    # Driving: stop
    if drive.driving:
        # Normal stop by the user
        drive.stop_drive('manual')
    # Standing: start if possible
    else:
        # Ask the loop
        ok, why = drive.start_drive()
        # Refused: explain right away
        if not ok:
            # Red refusal text
            drive_status.value = '<span style="color:red">%s</span>' % REFUSALS.get(why, why)

# Dropdown change handler: load the chosen model
def on_model_change(change):
    # A model was selected
    if change['new']:
        # Request it; the loop thread loads it
        drive.request_model(change['new'])

# "Rescan" button handler
def on_refresh(_=None):
    # Keep the current selection if it still exists
    current = model_dd.value
    # Re-read the checkpoints in MODELS_DIR
    model_dd.options = list_models(MODELS_DIR)
    # Restore the selection, or none
    model_dd.value = current if current in [p for _, p in model_dd.options] else None

# Every control widget: changes are copied into the controller at once
for w in (formula_tb, max_speed_s, min_speed_s, kp_s, ki_s, kd_s, bias_s, slow_s, alpha_s, slew_s, maxout_s,
          reverse_tb, timeout_s):
    # React to value changes
    w.observe(apply_controls, names='value')
# Start with the current widget values
apply_controls()
# Connect the drive button
drive_btn.on_click(on_drive)
# Connect the rescan button
refresh_btn.on_click(on_refresh)
# React to dropdown changes
model_dd.observe(on_model_change, names='value')
# Open / close the CSV log with the button
record_tb.observe(lambda change: drive.set_recording(change['new']), names='value')
# Start the inference + control thread and its watchdog (inference starts once a model is loaded)
drive.start()
# Start the display thread
display_worker.start()

## 5. 推論速度比較（報告用）

在**車子不動**的情況下，對 `models_trt/` 內每個模型用同一張畫面量 latency，條件相同才能比較（P.80）。執行時會先停車並暫停推論迴圈，量完自動恢復。沒有鏡頭時用一張灰色畫面。

In [ ]:
# Measure the inference latency of every checkpoint on the same frame (robot standing still)
def benchmark(n=100, warmup=20):
    # Stop the robot and wait until the loop is idle (raises if it does not answer within 10 s)
    drive.pause()
    # Always resume the loop afterwards
    try:
        # One frame shared by all models (grey frame without a camera)
        frame = camera.read() if camera is not None else None
        # Fallback frame
        if frame is None:
            # Uniform grey 224x224 BGR frame
            frame = np.full((224, 224, 3), 128, np.uint8)
        # (latency, label, runtime) per model
        rows = []
        # Every checkpoint in MODELS_DIR
        for label, path in list_models(MODELS_DIR):
            # Load it
            m = load_model(path, device)
            # Same input as in the loop (lane-marked)
            x = cam_mod.preprocess(mark_lanes(frame), device, m.dtype)
            # Warm-up runs, not timed
            for _ in range(warmup):
                # Run the network
                m(x)
            # Make sure the warm-up finished on the GPU
            if device.type == 'cuda': torch.cuda.synchronize()
            # Start the timer
            t0 = time.time()
            # Timed runs
            for _ in range(n):
                # Run the network
                m(x)
            # Wait for the GPU before stopping the timer
            if device.type == 'cuda': torch.cuda.synchronize()
            # Mean milliseconds per inference
            ms = (time.time() - t0) * 1000.0 / n
            # Store the result
            rows.append((ms, label, m.kind))
            # Free the model before loading the next one
            del m
    # Resume the loop
    finally:
        # Let the loop run again
        drive.resume()
    # Print fastest first
    for ms, label, kind in sorted(rows):
        # Latency, FPS, runtime and label
        print('%8.2f ms  %6.1f FPS  [%s]  %s' % (ms, 1000.0 / ms, kind, label))
    # Results for further use
    return rows

# Uncomment the next line to run the benchmark
# benchmark()

## 6. 結束：停車並釋放相機

做成按鈕，**只有手動按下才會執行**。如果直接寫成 cell，Run All 會一路跑到這裡，相機剛開就被關掉。

按下後先停車並結束推論執行緒（最多 1 秒），再結束顯示執行緒，最後釋放相機。

In [ ]:
# Button that stops everything and releases the camera
end_btn = widgets.Button(description='結束：停車並釋放相機', button_style='warning',
                         layout=widgets.Layout(width='220px'))

# "End" button handler
def on_end(_=None):
    # Inference and motors first: stop the robot, end the loop and the watchdog (at most 1 s)
    loop_ended = drive.close(timeout=1.0)
    # Then the display thread
    display_ended = display_worker.stop(timeout=1.0)
    # Release the camera for other notebooks
    if camera is not None:
        # Stop the camera
        camera.stop()
    # No more driving
    drive_btn.disabled = True
    # The button can only be used once
    end_btn.disabled = True
    # Show that the camera is free (and whether a thread did not end in time)
    end_btn.description = '相機已釋放' if loop_ended and display_ended else '相機已釋放（有執行緒未結束）'

# Connect the end button
end_btn.on_click(on_end)
# Show it
display(end_btn)